In [0]:
%python
dbutils.widgets.text("catalog","vp_dp20040222")
catalog=dbutils.widgets.get("catalog")

In [0]:
-- Create entitlements table (mask and filter read from current user's row)
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.ops.user_entitlements') (
    user_email STRING,
    allowed_city STRING,
    can_view_pii BOOLEAN
);

-- Create evidence table for G4
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.ops.governance_evidence') (
    evidence_name STRING,
    observed_value STRING
);

In [0]:
-- G1: Grant account users only USE CATALOG + USE SCHEMA/SELECT on gold
GRANT USE CATALOG ON CATALOG IDENTIFIER(:catalog) TO `account users`;
GRANT USE SCHEMA, SELECT ON SCHEMA IDENTIFIER(:catalog || '.gold') TO `account users`;

In [0]:
-- G2: Mask phone in gold.dim_member
-- Real value if can_view_pii = TRUE, otherwise XXXXXX + last 4 digits
CREATE OR REPLACE FUNCTION ops.mask_phone(phone STRING)
RETURNS STRING
RETURN CASE
    WHEN EXISTS (
        SELECT 1 FROM ops.user_entitlements
        WHERE user_email = current_user()
        AND can_view_pii = TRUE
    ) THEN phone
    ELSE 'XXXXXX' || RIGHT(phone, 4)
END;

ALTER TABLE IDENTIFIER(:catalog || '.gold.dim_member')
ALTER COLUMN phone
SET MASK ops.mask_phone;

In [0]:
-- G3: Row filter on gold.agg_daily_station
-- Show rows whose city = allowed_city, or all rows if ALL; no entitlement row → no rows
CREATE OR REPLACE FUNCTION ops.filter_by_city(station_id STRING)
RETURNS BOOLEAN
RETURN
    EXISTS (
        SELECT 1 FROM ops.user_entitlements e
        WHERE e.user_email = current_user()
        AND (
            e.allowed_city = 'ALL'
            OR e.allowed_city = (
                SELECT s.city FROM gold.dim_station s
                WHERE s.station_id = station_id
            )
        )
    );

ALTER TABLE IDENTIFIER(:catalog || '.gold.agg_daily_station')
SET ROW FILTER ops.filter_by_city ON (station_id);

In [0]:
-- G4: Set entitlement to Jaipur/FALSE, capture evidence, then set ALL/FALSE
DELETE FROM IDENTIFIER(:catalog || '.ops.user_entitlements') WHERE user_email = current_user();
INSERT INTO IDENTIFIER(:catalog || '.ops.user_entitlements') VALUES (current_user(), 'Jaipur', FALSE);

-- Clear old evidence
DELETE FROM IDENTIFIER(:catalog || '.ops.governance_evidence')
WHERE evidence_name IN ('ROW_FILTER_ROW_COUNT_JAIPUR', 'MASKED_PHONE_M0001');

-- Capture row count (row filter applies → only Jaipur rows visible)
INSERT INTO IDENTIFIER(:catalog || '.ops.governance_evidence') (evidence_name, observed_value)
SELECT 'ROW_FILTER_ROW_COUNT_JAIPUR', CAST(COUNT(*) AS STRING)
FROM IDENTIFIER(:catalog || '.gold.agg_daily_station');

-- Capture masked phone (column mask applies → XXXXXX + last 4 digits)
INSERT INTO IDENTIFIER(:catalog || '.ops.governance_evidence') (evidence_name, observed_value)
SELECT 'MASKED_PHONE_M0001', phone
FROM IDENTIFIER(:catalog || '.gold.dim_member')
WHERE member_id = 'M0001' AND is_current = TRUE
LIMIT 1;

-- Set entitlement to ALL, FALSE (mask and filter must remain in place)
UPDATE IDENTIFIER(:catalog || '.ops.user_entitlements')
SET allowed_city = 'ALL', can_view_pii = FALSE
WHERE user_email = current_user();